# NVIDIA Nemotron Reasoning — LoRA Fine-Tuning

bf16 + gradient checkpointing + batch_size=1. No pip install needed.

**GPU:** RTX Pro 6000 (96GB) — model ~62GB bf16, training fits in remaining ~33GB

In [ ]:
import os, re, glob, sys, gc, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pure PyTorch replacement for mamba_ssm's Triton rmsnorm_fn
# (Triton can't compile on Blackwell due to ptxas-blackwell permission issue)
def rmsnorm_fn_pytorch(x, weight, bias=None, z=None, eps=1e-6, group_size=None, norm_before_gate=True):
    x_shape = x.shape
    if group_size is not None:
        x = x.view(-1, x.shape[-1] // group_size, group_size)
    x_float = x.float()
    variance = x_float.pow(2).mean(-1, keepdim=True)
    x_normed = (x_float * torch.rsqrt(variance + eps)).to(x.dtype)
    if group_size is not None:
        x_normed = x_normed.view(x_shape)
    if weight is not None:
        x_normed = x_normed * weight
    if bias is not None:
        x_normed = x_normed + bias
    if z is not None:
        if norm_before_gate:
            x_normed = x_normed * F.silu(z)
        else:
            x_normed = x_normed  # fallback
    return x_normed

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
    print(f"Compute capability: sm_{props.major}{props.minor}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
LORA_RANK = 32           # max allowed by competition
LORA_ALPHA = 64
NUM_EPOCHS = 1           # 1 epoch first, add more if time allows
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 16          # effective batch = 16
MAX_SEQ_LEN = 512        # all prompts are <510 chars — this covers everything
MAX_TRAIN_SAMPLES = None # use ALL data (only 9500 total, fits easily)

# Competition model ID (MUST match for submission)
BASE_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

OUTPUT_DIR = Path("/kaggle/working")
ADAPTER_DIR = OUTPUT_DIR / "adapter"

SYSTEM_PROMPT = (
    "You are an expert at solving logical reasoning puzzles including "
    "bit manipulation, algebra, and text encryption. "
    "Solve the problem step by step, then give your final answer inside \\boxed{}."
)

def format_example(prompt, answer=None):
    text = f"<extra_id_0>System\n{SYSTEM_PROMPT}\n<extra_id_1>User\n{prompt}\n<extra_id_1>Assistant\n"
    if answer is not None:
        text += f"\\boxed{{{answer}}}"
    return text

def extract_answer(text):
    matches = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", text)
    return matches[-1].strip() if matches else None

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────
train_csvs = glob.glob("/kaggle/input/**/train.csv", recursive=True)
print("Found:", train_csvs)
assert train_csvs

df = pd.read_csv(train_csvs[0])
df.columns = [c.strip().lower() for c in df.columns]
df = df.dropna(subset=["prompt", "answer"]).reset_index(drop=True)
df["answer"] = df["answer"].astype(str).str.strip()

rng = np.random.default_rng(42)
n_val = 200  # larger val set for better accuracy estimate
val_idx = set(rng.choice(len(df), size=n_val, replace=False))
mask = np.array([i in val_idx for i in range(len(df))])
train_df = df[~mask].reset_index(drop=True)
val_df = df[mask].reset_index(drop=True)

if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(n=MAX_TRAIN_SAMPLES, random_state=42).reset_index(drop=True)

print(f"Total: {len(df)} | Train: {len(train_df)} | Val: {len(val_df)}")
print(f"Avg prompt: {df['prompt'].str.len().mean():.0f} chars | Max: {df['prompt'].str.len().max():.0f} chars")
print(f"Avg answer: {df['answer'].astype(str).str.len().mean():.1f} chars")

In [ ]:
# ── Clear GPU (run before reloading model) ─────────────────────────────
try: del model, trainer
except NameError: pass
gc.collect()
torch.cuda.empty_cache()
used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {used:.1f} / {total:.1f} GB ({total - used:.1f} GB free)")

In [ ]:
# ── Load Model + LoRA (bf16, FORCE pure PyTorch Mamba path) ───────────
utility_paths = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*")
if utility_paths:
    sys.path.insert(0, utility_paths[0])
    print(f"Utility script: {utility_paths[0]}")

try:
    import mamba_ssm
    # Patch rmsnorm_fn BEFORE model loads
    import mamba_ssm.ops.triton.layernorm_gated as ln_module
    ln_module.rmsnorm_fn = rmsnorm_fn_pytorch
    print("mamba_ssm loaded + rmsnorm_fn patched to pure PyTorch")
except Exception as e:
    print(f"mamba_ssm warning: {e}")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# Find model
candidates = glob.glob("/kaggle/input/**/config.json", recursive=True)
nemotron = [c for c in candidates if "nemotron" in c.lower()]
if nemotron:
    model_path = str(Path(nemotron[0]).parent)
else:
    import kagglehub
    model_path = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Model path: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (bf16)...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map={"":"cuda:0"},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print(f"Model loaded: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# CRITICAL: Force pure PyTorch Mamba (Blackwell CUDA kernels don't work)
for mod_name, mod in sys.modules.items():
    if "modeling_nemotron_h" in mod_name:
        if hasattr(mod, "is_fast_path_available"):
            mod.is_fast_path_available = False
        if hasattr(mod, "rmsnorm_fn"):
            mod.rmsnorm_fn = rmsnorm_fn_pytorch
        print(f"Patched {mod_name}: pure PyTorch mode")
        break

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Use competition's exact target modules
TARGET_MODULES = ["in_proj", "out_proj", "up_proj", "down_proj"]
print(f"LoRA targets: {TARGET_MODULES}")

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Sanity forward pass
print("Sanity forward pass...")
test_ids = tokenizer("Test", return_tensors="pt").to("cuda:0")
with torch.no_grad():
    _ = model(**test_ids)
print("Forward pass OK!")
print(f"GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB used")

In [ ]:
# ── Tokenize & Build Datasets ────────────────────────────────────────────
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

train_texts = [format_example(r["prompt"], r["answer"]) for _, r in train_df.iterrows()]
val_texts = [format_example(r["prompt"], r["answer"]) for _, r in val_df.iterrows()]

def tokenize_texts(texts):
    return tokenizer(texts, truncation=True, max_length=MAX_SEQ_LEN, padding=False, return_tensors=None)

train_tok = tokenize_texts(train_texts)
val_tok = tokenize_texts(val_texts)

train_ds = Dataset.from_dict({"input_ids": train_tok["input_ids"], "attention_mask": train_tok["attention_mask"]})
val_ds = Dataset.from_dict({"input_ids": val_tok["input_ids"], "attention_mask": val_tok["attention_mask"]})

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Avg tokens: {np.mean([len(ids) for ids in train_tok['input_ids']]):.0f}")

In [ ]:
# ── Train + auto-save every 100 steps ─────────────────────────────────────
import json
from transformers import TrainerCallback

class AutoSaveCallback(TrainerCallback):
    """Saves submission-ready adapter every N steps."""
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step > 0 and state.global_step % 100 == 0:
            self._save(model, state.global_step)
    def on_train_end(self, args, state, control, model=None, **kwargs):
        self._save(model, state.global_step)
    def _save(self, model, step):
        import subprocess
        save_dir = Path("/kaggle/working")
        model.save_pretrained(str(save_dir))
        tokenizer.save_pretrained(str(save_dir))
        cfg_path = save_dir / "adapter_config.json"
        cfg = json.loads(cfg_path.read_text())
        cfg["base_model_name_or_path"] = BASE_MODEL_ID
        cfg_path.write_text(json.dumps(cfg, indent=2))
        # Zip just the adapter files
        subprocess.run(
            f"cd {save_dir} && zip -r submission.zip adapter_config.json adapter_model.safetensors",
            shell=True
        )
        print(f"\n>> Step {step}: submission.zip saved. Click 'Save Version' when ready to submit.")

run_name = f"nemotron_r{LORA_RANK}_a{LORA_ALPHA}_ep{NUM_EPOCHS}"
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    run_name=run_name,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=data_collator,
    callbacks=[AutoSaveCallback()],
)

steps = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Training: {run_name}")
print(f"Samples: {len(train_ds)} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")
print(f"Steps/epoch: ~{steps} | Auto-saves every 100 steps + at end")
t0 = time.time()
trainer.train()
print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Quick Eval (20 samples — keep it fast) ───────────────────────────────
N_EVAL = 20
model.eval()
correct = no_answer = 0
t0 = time.time()

with torch.inference_mode():
    for i, (_, row) in enumerate(val_df.head(N_EVAL).iterrows()):
        enc = tokenizer(format_example(row["prompt"]), return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to("cuda:0")
        gen = model.generate(**enc, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        output = tokenizer.decode(gen[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = extract_answer(output)
        gt = str(row["answer"]).strip()
        if pred is None: no_answer += 1
        elif pred == gt: correct += 1

accuracy = correct / N_EVAL
elapsed = time.time() - t0
print(f"Val Accuracy: {accuracy:.4f} ({correct}/{N_EVAL}) | No-answer: {no_answer} | {elapsed:.0f}s")
print(f"Target: 0.683 | Gap: {0.683 - accuracy:+.4f}")

In [ ]:
# ── Save & Create Submission ─────────────────────────────────────────────
import json

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

# Fix base_model_name_or_path to HuggingFace ID (not local path)
cfg_path = ADAPTER_DIR / "adapter_config.json"
cfg = json.loads(cfg_path.read_text())
cfg["base_model_name_or_path"] = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
cfg_path.write_text(json.dumps(cfg, indent=2))
print("Fixed base_model_name_or_path in adapter_config.json")

os.chdir(str(ADAPTER_DIR))
subprocess.run("zip -r /kaggle/working/submission.zip adapter_config.json adapter_model.safetensors", shell=True, check=True)
os.chdir(str(OUTPUT_DIR))

print(f"submission.zip: {Path('/kaggle/working/submission.zip').stat().st_size / 1e6:.1f} MB")
print("Done! Go to Output tab -> Submit to competition.")